In [ ]:
import spacy
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings("ignore")

# Install gensim if not already installed, and then import
try:
    import gensim
    from gensim import corpora, models
except ImportError:
    print("gensim not found, installing...")
    !pip install gensim
    import gensim
    from gensim import corpora, models

# Load spaCy model
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("\n--- WARNING: spaCy model 'en_core_web_sm' not found. ---")
    print("Please run 'python -m spacy download en_core_web_sm' and restart the script.")
    # Exit or use a mock nlp object if execution environment allows
    raise

# Ensure VADER lexicon is available for Sentiment Analysis
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    print("Downloading VADER lexicon for NLTK...")
    nltk.download('vader_lexicon')

# Sample text data for demonstration
sample_data = [
    "The new iPhone 16 launched by Apple in California yesterday is truly revolutionary and fast.",
    "I regret buying this item; the quality is surprisingly poor and the customer support was terrible.",
    "Market analysis suggests a moderate rise in technology stocks next quarter, according to Bloomberg.",
    "This movie was neither great nor bad; the pacing was slow, but the cinematography was technically proficient.",
    "Global warming is a critical issue that demands immediate attention from governments worldwide.",
    "The engineers at Tesla are working on an innovative battery design in Fremont."
]

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [ ]:

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("\n--- WARNING: spaCy model 'en_core_web_sm' not found. ---")
    print("Please run 'python -m spacy download en_core_web_sm' and restart the script.")
    # Exit or use a mock nlp object if execution environment allows
    raise

# Ensure VADER lexicon is available for Sentiment Analysis
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except nltk.downloader.DownloadError:
    print("Downloading VADER lexicon for NLTK...")
    nltk.download('vader_lexicon')

# Sample text data for demonstration
sample_data = [
    "The new iPhone 16 launched by Apple in California yesterday is truly revolutionary and fast.",
    "I regret buying this item; the quality is surprisingly poor and the customer support was terrible.",
    "Market analysis suggests a moderate rise in technology stocks next quarter, according to Bloomberg.",
    "This movie was neither great nor bad; the pacing was slow, but the cinematography was technically proficient.",
    "Global warming is a critical issue that demands immediate attention from governments worldwide.",
    "The engineers at Tesla are working on an innovative battery design in Fremont."
]



In [ ]:
# --- LOW-LEVEL TEXT UNDERSTANDING (spaCy) ---

def perform_low_level_nlp(text):
    """
    Demonstrates Part-of-Speech (POS) Tagging, Named Entity Recognition (NER),
    and Dependency Parsing using spaCy.
    """
    print(f"\n\n\n############################################")
    print(f"### LOW-LEVEL NLP FOR: '{text[:50]}...' ")
    print(f"############################################")

    doc = nlp(text)

    # 1. Part-of-Speech (POS) Tagging
    print("\n--- 1. Part-of-Speech (POS) Tagging ---")
    pos_results = []
    for token in doc:
        # token.pos_ is the simple tag (NOUN, VERB, ADJ)
        # token.tag_ is the detailed tag (NN, VBD, JJ)
        pos_results.append((token.text, token.pos_, token.tag_, token.lemma_))
    print(pd.DataFrame(pos_results, columns=['Token', 'POS', 'Tag', 'Lemma']).to_string(index=False))

    # 2. Named Entity Recognition (NER)
    print("\n--- 2. Named Entity Recognition (NER) ---")
    ner_results = []
    for ent in doc.ents:
        # ent.label_ is the entity type (ORG, GPE, DATE)
        ner_results.append((ent.text, ent.label_, spacy.explain(ent.label_)))

    if ner_results:
        print(pd.DataFrame(ner_results, columns=['Entity', 'Label', 'Explanation']).to_string(index=False))
    else:
        print("No Named Entities found.")

    # 3. Dependency Parsing (Sentence Structure)
    print("\n--- 3. Dependency Parsing (Sentence Structure) ---")
    dep_results = []
    for token in doc:
        # token.dep_ is the type of dependency relation (nsubj, dobj, amod)
        # token.head.text is the word that this token modifies or is governed by
        dep_results.append((token.text, token.dep_, token.head.text, token.head.pos_))
    print(pd.DataFrame(dep_results, columns=['Token', 'Dependency', 'Head Token', 'Head POS']).to_string(index=False))

In [ ]:
# --- HIGH-LEVEL TEXT MINING (Sentiment Classification) ---

def perform_sentiment_analysis(texts):
    """
    Performs text classification (Sentiment Analysis) using VADER.
    VADER is a lexicon and rule-based sentiment analysis tool.
    """
    sid = SentimentIntensityAnalyzer()

    print("\n\n\n############################################")
    print("### HIGH-LEVEL NLP: SENTIMENT ANALYSIS (Classification) ###")
    print("############################################")

    results = []
    for text in texts:
        vs = sid.polarity_scores(text)

        # Simple classification logic:
        if vs['compound'] >= 0.05:
            sentiment_label = 'Positive'
        elif vs['compound'] <= -0.05:
            sentiment_label = 'Negative'
        else:
            sentiment_label = 'Neutral'

        results.append({
            'Text': text[:70] + '...',
            'Sentiment': sentiment_label,
            'Compound Score': f"{vs['compound']:.4f}",
            'Positive': f"{vs['pos']:.3f}",
            'Neutral': f"{vs['neu']:.3f}",
            'Negative': f"{vs['neg']:.3f}"
        })

    df = pd.DataFrame(results)
    print(df.to_string(index=False))

In [ ]:
# --- HIGH-LEVEL TEXT MINING (Topic Modeling) ---

def perform_topic_modeling(texts, num_topics=3):
    """
    Performs Topic Modeling using Latent Dirichlet Allocation (LDA) via gensim.
    This extracts the latent "topics" or themes from the texts.
    """

    print("\n\n\n############################################")
    print("### HIGH-LEVEL NLP: TOPIC MODELING (LDA) ###")
    print("############################################")

    # 1. Preprocessing (Tokenization, Lemmatization, Stopword Removal)
    def preprocess(text):
        doc = nlp(text.lower())
        # Filter out stop words, punctuation, and non-alphabetic tokens, then use lemma
        tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct and token.is_alpha]
        return tokens

    processed_docs = [preprocess(text) for text in texts]

    # Handle cases with very few unique documents or if preprocessing yields no tokens
    if len(processed_docs) < 2 or all(not doc for doc in processed_docs):
        print("Not enough unique documents or valid tokens to model topics effectively (need at least 2 non-empty documents).")
        return

    # 2. Create Dictionary (mapping of words to IDs) and Corpus (Bag-of-Words representation)
    dictionary = corpora.Dictionary(processed_docs)

    # Filter out tokens that appear in less than 1 document (i.e., appear in only one document or more)
    # and filter out tokens that appear in more than 90% of the documents
    # Relaxing no_below to 1 for smaller datasets.
    dictionary.filter_extremes(no_below=1, no_above=0.9)

    # If dictionary is empty after filtering, it means no terms were suitable
    if not dictionary:
        print("No terms left after dictionary filtering to build a topic model.")
        return

    corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

    # Check if the corpus is effectively empty after conversion to BOW
    if not corpus or all(not doc_bow for doc_bow in corpus):
        print("Corpus is empty or contains no terms after dictionary mapping. Cannot perform topic modeling.")
        return

    # 3. Apply LDA Model
    # Suppress gensim warnings for cleaner output
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')

        lda_model = models.LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=num_topics,
            random_state=42, # for reproducibility
            passes=15,
            alpha='auto'
        )

    # 4. Display Topics and Top Keywords
    print(f"\nDiscovered {num_topics} Topics (Top 5 keywords for each):")
    topics = lda_model.show_topics(formatted=False)

    topic_results = []
    for idx, topic in topics:
        keywords = [word[0] for word in topic[:5]]
        topic_results.append([f"Topic {idx+1}", ", ".join(keywords)])

    print(pd.DataFrame(topic_results, columns=['Topic ID', 'Keywords']).to_string(index=False))

    # 5. Document-Topic Distribution (showing which topic dominates each document)
    print("\n--- Document-Topic Distribution ---")
    doc_topic_results = []
    for i, doc_bow in enumerate(corpus):
        # Get the main topic for the document
        topics_per_doc = lda_model.get_document_topics(doc_bow, minimum_probability=0.1)

        if topics_per_doc:
            # Find the dominant topic (highest probability) a
            dominant_topic = max(topics_per_doc, key=lambda x: x[1])
            topic_id = dominant_topic[0] + 1
            probability = f"{dominant_topic[1]:.3f}"
        else:
            topic_id = "N/A"
            probability = "N/A"

        doc_topic_results.append({
            'Text': texts[i][:50] + '...',
            'Dominant Topic': f"Topic {topic_id}",
            'Probability': probability
        })

    print(pd.DataFrame(doc_topic_results).to_string(index=False))

In [ ]:
# --- EXECUTION ---

if __name__ == "__main__":

    # Run low-level NLP on the first example text
    perform_low_level_nlp(sample_data[0])

    # Run sentiment analysis on all texts
    perform_sentiment_analysis(sample_data)

    # Run topic modeling on all texts
    perform_topic_modeling(sample_data, num_topics=3)




############################################
### LOW-LEVEL NLP FOR: 'The new iPhone 16 launched by Apple in California ...' 
############################################

--- 1. Part-of-Speech (POS) Tagging ---
        Token   POS Tag         Lemma
          The   DET  DT           the
          new   ADJ  JJ           new
       iPhone PROPN NNP        iPhone
           16   NUM  CD            16
     launched  VERB VBN        launch
           by   ADP  IN            by
        Apple PROPN NNP         Apple
           in   ADP  IN            in
   California PROPN NNP    California
    yesterday  NOUN  NN     yesterday
           is   AUX VBZ            be
        truly   ADV  RB         truly
revolutionary   ADJ  JJ revolutionary
          and CCONJ  CC           and
         fast   ADJ  JJ          fast
            . PUNCT   .             .

--- 2. Named Entity Recognition (NER) ---
    Entity Label                             Explanation
    iPhone   ORG Companies, agencies, in